In [2]:
import json
import os
from Tabelog_scraper import get_detail_info, jp_to_en_category  # reuse your existing module

# --- Config ---
TARGET_URL = "https://tabelog.com/hyogo/A2807/A280703/28001965/"
CATEGORY_JP = "スイーツ"
CATEGORY_EN = jp_to_en_category(CATEGORY_JP)
REGION = "WEST"
CATEGORY_FILE = f"json_exports/tabelog_{CATEGORY_EN}_{REGION}_2025.json"

# --- Make sure file exists ---
if not os.path.exists(CATEGORY_FILE):
    raise FileNotFoundError(f"{CATEGORY_FILE} not found — run the main scraper first.")

# --- Load existing JSON ---
with open(CATEGORY_FILE, "r", encoding="utf-8") as f:
    data = json.load(f)

# --- Skip if already present ---
if any(r.get("url") == TARGET_URL for r in data):
    print(f"✅ Restaurant already present in {CATEGORY_FILE}")
    exit(0)

# --- Extract area info (from URL path) ---
area_text = "Hyogo"  # you can refine this if desired

# --- Call detail scraper ---
print(f"Scraping single restaurant: {TARGET_URL}")
(
    rating,
    price_dinner,
    price_lunch,
    business_items,
    hours_notes_structured,
    name_en,
    name_jp,
    g_info,
    sub_categories,
    review_count,
    g_rating,
    g_reviews,
    google_maps_url,
) = get_detail_info(TARGET_URL, area_text)

# --- Build record ---
record = {
    "category_jp": CATEGORY_JP,
    "category_en": CATEGORY_EN,
    "region": REGION,
    "name": name_jp or name_en,
    "name_en": name_en,
    "area": area_text,
    "url": TARGET_URL,
    "rating": rating,
    "review_count_tabelog": review_count,
    "price_dinner": price_dinner,
    "price_lunch": price_lunch,
    "sub_categories": ", ".join(sub_categories),
    "hours_raw": business_items,
    "hours_notes_structured": hours_notes_structured,
    "google_rating": g_rating,
    "google_reviews": g_reviews,
    "google_maps_url": google_maps_url,
    "image_url": "",
    **g_info,
}

# --- Append and save ---
data.append(record)
with open(CATEGORY_FILE, "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print(f"✅ Added '{name_en or name_jp}' to {CATEGORY_FILE} (total now {len(data)} entries)")


Scraping single restaurant: https://tabelog.com/hyogo/A2807/A280703/28001965/
✅ Added 'Pâtissier Es Koyama' to json_exports/tabelog_Sweets_WEST_2025.json (total now 100 entries)
